# Simplicits Debug on Surgical Tissue Point Cloud (Gravity Only)

This notebook loads a **surface point cloud** from your pickle file and runs a **first-pass Simplicits simulation under gravity** (no tool constraints yet).

**Notes**
- This follows the structure of Kaolin's Simplicits Easy API example.
- If you run in **VS Code** and the Kaolin ipywidget visualizer fails, use the **k3d** visualization cells (they work in most environments).


In [1]:
# --- Imports ---
import os
import pickle
import numpy as np
import torch

import kaolin as kal

# For point cloud visualization (recommended for VS Code + Jupyter)
#   pip install k3d
import k3d

## 1) Load tissue point cloud from PKL

In [2]:
pkl_file_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tissue_pts_dnsampled_once.pkl"

with open(pkl_file_path, "rb") as f:
    data = pickle.load(f)

print(type(data), len(data))
print("keys:", data[0].keys())
print("xyz:", np.asarray(data[0]["xyz"]).shape)

<class 'list'> 200
keys: dict_keys(['frame_id', 'xyz', 'rgb', 'opacity', 'indices'])
xyz: (38426, 3)


## 2) Choose a rest frame and move to GPU

In [3]:
# Choose which frame to treat as rest state
rest_frame_idx = 0

xyz = np.asarray(data[rest_frame_idx]["xyz"], dtype=np.float32)  # (N,3)
N = xyz.shape[0]
print("N =", N)

pts = torch.from_numpy(xyz).cuda()

# Center + normalize into roughly [-1,1] cube (helps training stability)
pts = kal.ops.pointcloud.center_points(pts.unsqueeze(0), normalize=True).squeeze(0)

orig_pts = pts.clone()

N = 38426


## 3) Quick visualization of rest point cloud (k3d)

In [11]:
plot = k3d.plot()
k3d_pts = k3d.points(orig_pts.detach().cpu().numpy(), point_size=0.01)
plot += k3d_pts
plot.display()

Output()

## 4) Material fields (constant for now)

In [5]:
# These are *per-point* material fields used by the elastic loss / simulation.
# Start simple: constant fields.
# Units are not super important for this debug step; tune later.

yms  = torch.full((N,), 2e5, device=pts.device, dtype=pts.dtype)   # Young's modulus
prs  = torch.full((N,), 0.45, device=pts.device, dtype=pts.dtype)  # Poisson ratio
rhos = torch.full((N,), 1000., device=pts.device, dtype=pts.dtype) # Density

# Approx volume: for surface point clouds this is not "true volume".
# Use a rough proxy based on bounding box volume to get reasonable scaling.
mn = pts.min(dim=0).values
mx = pts.max(dim=0).values
bbox_vol = float(torch.prod(mx - mn).detach().cpu())
approx_volume = max(bbox_vol, 1e-6)

print("approx_volume (bbox proxy):", approx_volume)

approx_volume (bbox proxy): 0.6332738995552063


## 5) Train a SimplicitsObject (learn weights)

This step learns the **skinning weight field** \(w(x)\) (self-supervised) using elastic energy under random handle transforms.

Start with low iterations to debug the pipeline; increase later.


In [6]:
# Handles = reduced DOFs. Start small for debugging.
num_handles = 5

# Tip: increase training_num_steps after the first end-to-end run works.
sim_obj = kal.physics.simplicits.SimplicitsObject.create_trained(
    pts,
    yms,
    prs,
    rhos,
    approx_volume,
    num_handles=num_handles,

    training_num_steps=3000,
    training_lr_start=1e-3,
    training_lr_end=1e-3,

    # Coeffs similar to Kaolin example; adjust if training unstable.
    training_le_coeff=1e-1,
    training_lo_coeff=1e6,

    training_log_every=500,
    normalize_for_training=True,
)

print("trained object:", sim_obj)

trained object: <kaolin.physics.simplicits.easy_api.SimplicitsObject object at 0x7f920ca01030>


## 6) Build a scene and enable gravity-only simulation

In [9]:
scene = kal.physics.simplicits.SimplicitsScene()

# Debug-friendly settings
scene.max_newton_steps = 8
scene.timestep = 0.02
scene.direct_solve = True

obj_idx = scene.add_object(sim_obj)

# Kaolin convention: world_up_axis=1 => gravity along +Y in their example.
# If you want gravity downward, flip sign.
scene.set_scene_gravity(acc_gravity=torch.tensor([0.0, 9.8, 0.0], device=pts.device, dtype=pts.dtype))

# No floor yet (debug). Add later once motion looks sane.
# scene.set_scene_floor(floor_height=-0.8, floor_axis=1, floor_penalty=1000.0)

scene.reset_scene()

## 7) Run a few sim steps and update the point cloud in k3d

In [10]:
def get_deformed_points():
    # Returns (N,3) torch tensor on GPU
    return scene.get_object_deformed_pts(obj_idx, orig_pts)

# Warm-up fetch
deformed = get_deformed_points()
k3d_pts.positions = deformed.detach().cpu().numpy()

# Step the simulator
num_steps = 200

for s in range(num_steps):
    scene.run_sim_step()
    if s % 2 == 0:
        deformed = get_deformed_points()
        k3d_pts.positions = deformed.detach().cpu().numpy()

## 8) Next: add a simple attachment constraint (preview)

Once gravity-only works, the simplest first constraint is a **soft attachment penalty** on a subset of points.

Conceptually:
\[
E_{attach}(z)=\frac{k}{2}\sum_{i\in\mathcal{I}}\|x_i(z)-x_i^*\|^2
\]

Implementation depends on whether you attach in:
- vertex/point space (easy if you can add an energy term), or
- handle space by modifying z directly (less physical).

If you paste the specific scene API you’re using to add boundary/penalty constraints (or show the functions in your `easy_api.py`), we can wire it in cleanly.
